## Imports and Configs

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.append(str(REPO_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt
import requests
import json
import time
from transformer_lens import HookedTransformer
from huggingface_hub import login
from sae_lens import SAE

DATA_DIR    = REPO_ROOT / "data"
FIGURES_DIR = REPO_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## Loggin into Huggingface

In [2]:
login()

In [ ]:
# Load the IOI dataset (clean + corrupted)
pilot = torch.load(DATA_DIR / "ioi_dataset.pt", weights_only=False)
clean_toks       = pilot["clean_toks"].to(device)
corrupted_toks   = pilot["corrupted_toks"].to(device)
corrupted_io_ids = pilot["corrupted_io_token_ids"].to(device)
corrupted_s_ids  = pilot["corrupted_s_token_ids"].to(device)
end_positions    = pilot["end_positions"].to(device)
N                = pilot["N"]

# Load the PC-learned graph + the binarized feature mat
graph_data = torch.load(DATA_DIR / "pc_learned_graph.pt", weights_only=False)
edges        = graph_data["edges"]          
col_to_layer = graph_data["col_to_layer"]
col_to_feat  = graph_data["col_to_feat"]
selected     = graph_data["selected"]        
LAYERS       = graph_data["layers"]          
X_binary     = graph_data["X_binary"]      

n_features = len(col_to_layer)
print(f"N prompts: {N}")
print(f"PC graph: {n_features} features, {len(edges)} edges")
print(f"Layers: {LAYERS}")


N prompts: 5000
PC graph: 115 features, 366 edges
Layers: [18, 22, 25]


## Loading in Gemma

In [4]:
print("Loading Gemma-2-2B...")
model = HookedTransformer.from_pretrained_no_processing(
    "gemma-2-2b", device=device, dtype=torch.bfloat16,
)
model.eval()
print(f"Model loaded. d_model={model.cfg.d_model}, n_layers={model.cfg.n_layers}")

print("\nLoading SAEs at layers", LAYERS)
saes = {}
for L in LAYERS:
    sae, _, _ = SAE.from_pretrained(
        release="gemma-scope-2b-pt-res-canonical",
        sae_id=f"layer_{L}/width_16k/canonical",
        device=device,
    )
    sae.eval()
    saes[L] = sae
    print(f"  Layer {L} SAE loaded. d_sae = {sae.cfg.d_sae}")

print(f"\nVRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


Loading Gemma-2-2B...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b into HookedTransformer
Model loaded. d_model=2304, n_layers=26

Loading SAEs at layers [18, 22, 25]


/tmp/ipykernel_21510/3050373775.py:11: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, _, _ = SAE.from_pretrained(


  Layer 18 SAE loaded. d_sae = 16384
  Layer 22 SAE loaded. d_sae = 16384
  Layer 25 SAE loaded. d_sae = 16384

VRAM allocated: 9.28 GB


In [ ]:
def ioi_logit_diff(logits, io_ids, s_ids, ends):
    #Same metric: logit of IO minus logit of S at the end token, sum (not mean) over the batch like the original paper
    batch_idx    = torch.arange(logits.shape[0], device=logits.device)
    final_logits = logits[batch_idx, ends]
    io_l = final_logits.gather(1, io_ids.unsqueeze(1)).squeeze(1)
    s_l  = final_logits.gather(1, s_ids.unsqueeze(1)).squeeze(1)
    return (io_l - s_l).sum()

 #Hooking resid_post at just our 3 target layers (18, 22, 25), the only layers we actually have SAEs for
HOOK_NAMES = [f"blocks.{L}.hook_resid_post" for L in LAYERS]
BATCH_SIZE = 32

#One d_sae-sized accumulator per layer, we'll sum into these across batches and divide by N at the end
attribution_sum = {L: torch.zeros(saes[L].cfg.d_sae, device=device, dtype=torch.float32)
                   for L in LAYERS}

print(f"Running per-feature AP in batches of {BATCH_SIZE}...")

Running per-feature AP in batches of 32...


In [ ]:
for i in range(0, N, BATCH_SIZE):
    j = min(i + BATCH_SIZE, N)
    bsz = j - i
    ends = end_positions[i:j]

    #clean forward, cache resid_post at our target layers (no grad)
    clean_resid = {}
    #Factory pattern so each hook closure remembers which layer L it's for
    def make_hook(L):
        def hook_fn(act, hook):
            clean_resid[L] = act.detach()
            return act
        return hook_fn
    with torch.no_grad():
        _ = model.run_with_hooks(
            clean_toks[i:j],
            fwd_hooks=[(name, make_hook(L)) for name, L in zip(HOOK_NAMES, LAYERS)],
            return_type=None,
        )

    #corrupted forward, same idea
    corrupted_resid = {}
    def make_corr_hook(L):
        def hook_fn(act, hook):
            corrupted_resid[L] = act.detach()
            return act
        return hook_fn
    with torch.no_grad():
        _ = model.run_with_hooks(
            corrupted_toks[i:j],
            fwd_hooks=[(name, make_corr_hook(L)) for name, L in zip(HOOK_NAMES, LAYERS)],
            return_type=None,
        )

    #Encode the cached resid streams into SAE features at the END position only...
    #...coz that's where IOI happens, no need to bother with the other tokens
    batch_idx = torch.arange(bsz, device=device)
    clean_feats = {}
    corr_feats  = {}
    for L in LAYERS:
        c_end = clean_resid[L][batch_idx, ends]
        co_end = corrupted_resid[L][batch_idx, ends]
        #Cast to the SAE's dtype before encoding so we don't get a dtype mismatch
        clean_feats[L] = saes[L].encode(c_end.to(saes[L].W_enc.dtype))
        corr_feats[L]  = saes[L].encode(co_end.to(saes[L].W_enc.dtype))


    #corrupted forward + backward, but with the SAE encode/decode splice...
    #...this is the trick that lets us grab gradients w.r.t. SAE features and not just the resid stream
    feat_grads = {}
    def make_grad_hook(L):
        def hook_fn(act, hook):
            ends_local = end_positions[i:j]
            batch_idx_local = torch.arange(bsz, device=device)
            #Pull the end-position resid, encode to features, mark for grad, decode, splice back...
            #...the rest of the forward pass then sees the SAE recon at the end position so grads flow through the features
            resid_end = act[batch_idx_local, ends_local]
            features = saes[L].encode(resid_end.to(saes[L].W_enc.dtype))
            features = features.detach().requires_grad_(True)
            features.retain_grad()
            feat_grads[L] = features
            recon_end = saes[L].decode(features).to(act.dtype)
            act[batch_idx_local, ends_local] = recon_end
            return act
        return hook_fn

    cor_logits_grad = model.run_with_hooks(
        corrupted_toks[i:j],
        fwd_hooks=[(name, make_grad_hook(L)) for name, L in zip(HOOK_NAMES, LAYERS)],
    )
    metric = ioi_logit_diff(cor_logits_grad,
                            corrupted_io_ids[i:j], corrupted_s_ids[i:j], ends)
    metric.backward()
    del cor_logits_grad, metric

    #Same attribution math  (diff * grad, summed over batch) but on SAE features instead of resid stream
    for L in LAYERS:
        diff = (clean_feats[L] - corr_feats[L]).float()
        grad = feat_grads[L].grad.detach().float()
        attribution_sum[L] += (diff * grad).sum(dim=0)

    #Free everything coz GPU poor
    del clean_resid, corrupted_resid, clean_feats, corr_feats, feat_grads
    torch.cuda.empty_cache()

    if (i // BATCH_SIZE) % 16 == 0:
        print(f"  batch {i//BATCH_SIZE + 1}/{(N + BATCH_SIZE - 1)//BATCH_SIZE}  "
            f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

#Average over N prompts and move to CPU for cheap inspection
ap_per_feature = {L: (attribution_sum[L] / N).cpu() for L in LAYERS}
print("\nPer-feature AP done.")
for L in LAYERS:
    a = ap_per_feature[L]
    print(f"  Layer {L}: range [{a.min():+.3f}, {a.max():+.3f}], "
        f"top-5 abs values: {a.abs().topk(5).values.tolist()}")

  batch 1/157  VRAM: 16.2 GB
  batch 17/157  VRAM: 16.8 GB
  batch 33/157  VRAM: 17.3 GB
  batch 49/157  VRAM: 17.9 GB
  batch 65/157  VRAM: 18.4 GB
  batch 81/157  VRAM: 19.0 GB
  batch 97/157  VRAM: 19.5 GB
  batch 113/157  VRAM: 20.1 GB
  batch 129/157  VRAM: 20.6 GB
  batch 145/157  VRAM: 21.1 GB

Per-feature AP done.
  Layer 18: range [+0.000, +0.000], top-5 abs values: [0.0, 0.0, 0.0, 0.0, 0.0]
  Layer 22: range [+0.000, +0.000], top-5 abs values: [0.0, 0.0, 0.0, 0.0, 0.0]
  Layer 25: range [-0.320, +0.074], top-5 abs values: [0.31956368684768677, 0.16244323551654816, 0.140384241938591, 0.136159747838974, 0.13003221154212952]


In [ ]:
ap_for_selected = []
for col in range(n_features):
    L = col_to_layer[col]
    f_idx = col_to_feat[col]
    ap_for_selected.append(ap_per_feature[L][f_idx].item())

ap_for_selected = torch.tensor(ap_for_selected)
print(f"AP scores for the 115 PC-selected features:")
print(f"  range: [{ap_for_selected.min():+.3f}, {ap_for_selected.max():+.3f}]")
print(f"  most negative (strongest 'this feature pushes toward IO'): "
      f"{ap_for_selected.min():+.3f}")
print(f"  most positive: {ap_for_selected.max():+.3f}")

torch.save({
    "ap_per_feature":   ap_per_feature,        
    "ap_for_selected":  ap_for_selected,        
    "col_to_layer":     col_to_layer,
    "col_to_feat":      col_to_feat,
}, DATA_DIR / "ap_per_feature.pt")
print(f"\nSaved per-feature AP to {DATA_DIR / 'ap_per_feature.pt'}")


AP scores for the 115 PC-selected features:
  range: [-0.320, +0.074]
  most negative (strongest 'this feature pushes toward IO'): -0.320
  most positive: +0.074

Saved per-feature AP to /workspace/Independence-Is-All-You-Need/data/ap_per_feature.pt


In [ ]:
def fetch_neuronpedia_label(layer: int, feature_idx: int, retries: int = 2) -> dict:

    sae_name = f"{layer}-gemmascope-res-16k"
    url = f"https://www.neuronpedia.org/api/feature/gemma-2-2b/{sae_name}/{feature_idx}"
    for attempt in range(retries + 1):
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                data = r.json()
                # Auto-interp explanation lives under 'explanations' (a list)
                expls = data.get("explanations", [])
                label = expls[0]["description"] if expls else None
                return {"layer": layer, "feature": feature_idx, "label": label, "raw": data}
            elif r.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            else:
                return {"layer": layer, "feature": feature_idx, "label": None,
                        "raw": {"error": f"HTTP {r.status_code}"}}
        except Exception as e:
            if attempt < retries:
                time.sleep(1)
                continue
            return {"layer": layer, "feature": feature_idx, "label": None,
                    "raw": {"error": str(e)}}
    return {"layer": layer, "feature": feature_idx, "label": None, "raw": None}

# Fetch labels for the 115 selected features
print(f"Fetching Neuronpedia labels for {n_features} features...")
labels = []
for col in range(n_features):
    L = col_to_layer[col]
    f_idx = col_to_feat[col]
    info = fetch_neuronpedia_label(L, f_idx)
    labels.append(info)
    if (col + 1) % 20 == 0:
        n_labeled = sum(1 for x in labels if x["label"])
        print(f"  fetched {col+1}/{n_features}, {n_labeled} have labels")
    time.sleep(0.05)   # be polite to the API

n_labeled = sum(1 for x in labels if x["label"])
print(f"\nDone. {n_labeled}/{n_features} features have auto-interp labels.")


Fetching Neuronpedia labels for 115 features...
  fetched 20/115, 20 have labels
  fetched 40/115, 40 have labels
  fetched 60/115, 60 have labels
  fetched 80/115, 80 have labels
  fetched 100/115, 100 have labels

Done. 115/115 features have auto-interp labels.


In [ ]:
# Save labels
with open(DATA_DIR / "neuronpedia_labels.json", "w") as f:

    save_labels = [{"layer": x["layer"], "feature": x["feature"], "label": x["label"]}
                   for x in labels]
    json.dump(save_labels, f, indent=2)
print(f"Saved labels to {DATA_DIR / 'neuronpedia_labels.json'}")


print("\nSample labels per layer:")
for L in LAYERS:
    print(f"\n  Layer {L}:")
    layer_labels = [(col, x) for col, x in enumerate(labels)
                    if col_to_layer[col] == L and x["label"]]
    for col, x in layer_labels[:3]:
        snippet = x["label"][:100] + ("..." if len(x["label"]) > 100 else "")
        print(f"    F{x['feature']}: {snippet}")


Saved labels to /workspace/Independence-Is-All-You-Need/data/neuronpedia_labels.json

Sample labels per layer:

  Layer 18:
    F13264: references to dog behaviors and interactions
    F3851: references to legal terms and proceedings
    F9378: instances of interpersonal conflict and emotional reactions in relationships

  Layer 22:
    F3497: references to personal health and medical conditions
    F10043: sentences that convey opinions or judgments about societal norms and personal relationships
    F15056: certain key terms and phrases related to various subjects such as programming, medicine, and science

  Layer 25:
    F14325:  numerical data and statistics related to incidents or events
    F12908: commands related to identity and credential management in a coding or command context
    F5012: themes related to government and qualifications for leadership


In [ ]:
adjacency = {i: set() for i in range(n_features)}
parents   = {i: set() for i in range(n_features)}
for p, c, _ in edges:
    parents[c].add(p)
    adjacency[p].add(c)
    adjacency[c].add(p)

v_structures = []
for c in range(n_features):
    parent_list = sorted(parents[c])
    for i, A in enumerate(parent_list):
        for B in parent_list[i+1:]:
            if B not in adjacency[A]:    
                v_structures.append((A, c, B))


cross_layer_vs = [(A, C, B) for A, C, B in v_structures
                  if col_to_layer[A] != col_to_layer[C] or col_to_layer[B] != col_to_layer[C]]

def is_labeled(col):
    return labels[col]["label"] is not None

def score_v_structure(A, C, B):
    score = 0

    if is_labeled(A) and is_labeled(B) and is_labeled(C):
        score += 10

    a_ap = abs(ap_for_selected[A].item())
    b_ap = abs(ap_for_selected[B].item())
    c_ap = abs(ap_for_selected[C].item())
    if a_ap > 0.05 and b_ap > 0.05 and c_ap > 0.1:    # all "matter" by AP
        score += 5

    if col_to_layer[A] == col_to_layer[B] and col_to_layer[C] > col_to_layer[A]:
        score += 5
    return score

scored = sorted(cross_layer_vs, key=lambda x: -score_v_structure(*x))
print(f"Top 10 candidate v-structures by interpretability heuristic:")
for rank, (A, C, B) in enumerate(scored[:10], 1):
    s = score_v_structure(A, C, B)
    L_A, L_B, L_C = col_to_layer[A], col_to_layer[B], col_to_layer[C]
    f_A, f_B, f_C = col_to_feat[A], col_to_feat[B], col_to_feat[C]
    label_A = (labels[A]["label"] or "[no label]")[:50]
    label_B = (labels[B]["label"] or "[no label]")[:50]
    label_C = (labels[C]["label"] or "[no label]")[:50]
    print(f"\n  #{rank} (score={s})")
    print(f"    A = L{L_A} F{f_A}: {label_A}")
    print(f"    B = L{L_B} F{f_B}: {label_B}")
    print(f"    C = L{L_C} F{f_C}: {label_C}")


Top 10 candidate v-structures by interpretability heuristic:

  #1 (score=15)
    A = L18 F14807: references to legal or formal documents and titles
    B = L18 F5748:  expressions of gratitude and appreciation
    C = L22 F6990: references to children and family dynamics

  #2 (score=15)
    A = L18 F12116: elements related to interpersonal conflicts and em
    B = L18 F61: references to significant women in history or prom
    C = L22 F1208: instances of significant financial outcomes or eve

  #3 (score=15)
    A = L18 F9378: instances of interpersonal conflict and emotional 
    B = L18 F6140: phrases and descriptions related to abilities and 
    C = L22 F4416: references to character dynamics and interrelation

  #4 (score=15)
    A = L18 F6140: phrases and descriptions related to abilities and 
    B = L18 F2297: names of prominent characters or actors associated
    C = L22 F4416: references to character dynamics and interrelation

  #5 (score=15)
    A = L18 F12116: elements r

In [11]:
# Pick the top-scoring v-structure for the case study
A, C, B = scored[0]
L_A, L_B, L_C = col_to_layer[A], col_to_layer[B], col_to_layer[C]
f_A, f_B, f_C = col_to_feat[A], col_to_feat[B], col_to_feat[C]

print(f"Selected v-structure for case study:")
print(f"  A = L{L_A} F{f_A} — {labels[A]['label']}")
print(f"  B = L{L_B} F{f_B} — {labels[B]['label']}")
print(f"  C = L{L_C} F{f_C} — {labels[C]['label']}")

print(f"\nAP scores (per-feature attribution to IOI metric):")
print(f"  A: {ap_for_selected[A]:+.4f}")
print(f"  B: {ap_for_selected[B]:+.4f}")
print(f"  C: {ap_for_selected[C]:+.4f}")


Selected v-structure for case study:
  A = L18 F14807 — references to legal or formal documents and titles
  B = L18 F5748 —  expressions of gratitude and appreciation
  C = L22 F6990 — references to children and family dynamics

AP scores (per-feature attribution to IOI metric):
  A: +0.0000
  B: +0.0000
  C: +0.0000


In [12]:
from scipy.stats import chi2_contingency

a_col, b_col, c_col = X_binary[:, A], X_binary[:, B], X_binary[:, C]

# Marginal contingency table A x B
marginal_table = np.zeros((2, 2), dtype=int)
for ai in [0, 1]:
    for bi in [0, 1]:
        marginal_table[ai, bi] = ((a_col == ai) & (b_col == bi)).sum()

chi2_marg, p_marg, _, _ = chi2_contingency(marginal_table)
print(f"Marginal A vs B:")
print(f"  contingency: {marginal_table.tolist()}")
print(f"  chi2 = {chi2_marg:.3f}, p = {p_marg:.4f}  "
      f"({'INDEPENDENT' if p_marg > 0.05 else 'dependent'})")

# Conditional A x B | C=0 and A x B | C=1
print(f"\nConditional A vs B given C:")
for c_val in [0, 1]:
    mask = (c_col == c_val)
    if mask.sum() < 20:
        print(f"  C={c_val}: too few samples ({mask.sum()})")
        continue
    cond_table = np.zeros((2, 2), dtype=int)
    for ai in [0, 1]:
        for bi in [0, 1]:
            cond_table[ai, bi] = ((a_col == ai) & (b_col == bi) & mask).sum()
    if (cond_table.sum(axis=0) == 0).any() or (cond_table.sum(axis=1) == 0).any():
        print(f"  C={c_val}: degenerate table {cond_table.tolist()}")
        continue
    chi2_cond, p_cond, _, _ = chi2_contingency(cond_table)
    verdict = 'INDEPENDENT' if p_cond > 0.05 else 'dependent'
    print(f"  C={c_val} (n={mask.sum()}): chi2={chi2_cond:.3f}, p={p_cond:.4f}  ({verdict})")

print("\nExpected v-structure signature: marginally independent, conditionally dependent")


Marginal A vs B:
  contingency: [[1242, 1218], [891, 1649]]
  chi2 = 120.674, p = 0.0000  (dependent)

Conditional A vs B given C:
  C=0 (n=1141): chi2=11.355, p=0.0008  (dependent)
  C=1 (n=3859): chi2=9.241, p=0.0024  (dependent)

Expected v-structure signature: marginally independent, conditionally dependent


In [ ]:
from scipy.stats import chi2_contingency
import numpy as np


print("Searching for v-structures with clean explaining-away signature...")
print("(marginal A⊥B, but A⊥B|C breaks)")

best = None
best_score = -np.inf

for A_cand, C_cand, B_cand in scored[:100]:
    if not (is_labeled(A_cand) and is_labeled(B_cand) and is_labeled(C_cand)):
        continue

    a = X_binary[:, A_cand]
    b = X_binary[:, B_cand]
    c = X_binary[:, C_cand]

    # Marginal A vs B
    try:
        marg = np.zeros((2, 2), int)
        for ai in [0, 1]:
            for bi in [0, 1]:
                marg[ai, bi] = ((a == ai) & (b == bi)).sum()
        if (marg.sum(0) == 0).any() or (marg.sum(1) == 0).any():
            continue
        _, p_marg, _, _ = chi2_contingency(marg)

        # Conditional A vs B | C=1
        mask = c == 1
        if mask.sum() < 50:
            continue
        cond = np.zeros((2, 2), int)
        for ai in [0, 1]:
            for bi in [0, 1]:
                cond[ai, bi] = ((a == ai) & (b == bi) & mask).sum()
        if (cond.sum(0) == 0).any() or (cond.sum(1) == 0).any():
            continue
        _, p_cond, _, _ = chi2_contingency(cond)

        # Score: prefer marginal indep + conditional dependence
        # Higher = better. Want p_marg high, p_cond low.
        if p_marg > 0.05 and p_cond < 0.05:
            score = p_marg - p_cond
            if score > best_score:
                best_score = score
                best = (A_cand, C_cand, B_cand, p_marg, p_cond)
                print(f"  Candidate: L{col_to_layer[A_cand]}F{col_to_feat[A_cand]}, "
                      f"L{col_to_layer[B_cand]}F{col_to_feat[B_cand]} → "
                      f"L{col_to_layer[C_cand]}F{col_to_feat[C_cand]}: "
                      f"p_marg={p_marg:.3f}, p_cond={p_cond:.3f}")
    except Exception as e:
        continue

if best is None:
    print("\nNo clean explaining-away v-structure found in top 100.")
    print("Falling back to top scored candidate.")
    A, C, B = scored[0]
else:
    A, C, B = best[0], best[1], best[2]
    print(f"\nSelected: marginal p={best[3]:.4f}, conditional p={best[4]:.4f}")

L_A, L_B, L_C = col_to_layer[A], col_to_layer[B], col_to_layer[C]
f_A, f_B, f_C = col_to_feat[A], col_to_feat[B], col_to_feat[C]
print(f"\nCase study v-structure:")
print(f"  A = L{L_A} F{f_A} — {labels[A]['label']}")
print(f"  B = L{L_B} F{f_B} — {labels[B]['label']}")
print(f"  C = L{L_C} F{f_C} — {labels[C]['label']}")

Searching for v-structures with clean explaining-away signature...
(marginal A⊥B, but A⊥B|C breaks)
  Candidate: L18F12116, L18F61 → L22F1208: p_marg=0.472, p_cond=0.021
  Candidate: L18F12116, L18F61 → L22F13888: p_marg=0.472, p_cond=0.000
  Candidate: L18F15143, L18F2297 → L22F10566: p_marg=0.603, p_cond=0.000

Selected: marginal p=0.6027, conditional p=0.0000

Case study v-structure:
  A = L18 F15143 —  instances of dialogue or reported speech
  B = L18 F2297 — names of prominent characters or actors associated with performances in films or television shows
  C = L22 F10566 — specific words and phrases related to arguments, reasoning, and the presentation of evidence in a legal or analytical context


In [ ]:
def get_feature_activation_at_end(toks, ends, layer, feat_idx, batch_size=64):
    #Helper to grab one specific SAE feature's activation at the END token across a whole prompt set...
    #...we'll use this to read out feature values both pre- and post-intervention
    activations = []
    sae = saes[layer]
    hook_name = f"blocks.{layer}.hook_resid_post"
    for i in range(0, len(toks), batch_size):
        b_toks = toks[i:i+batch_size]
        b_ends = ends[i:i+batch_size]
        cache = {}
        def hook_fn(act, hook):
            cache["resid"] = act
            return act
        with torch.no_grad():
            _ = model.run_with_hooks(b_toks, fwd_hooks=[(hook_name, hook_fn)],
                                    return_type=None)
        #Pull resid at the end token, encode through the SAE, keep only the feature we care about
        resid_end = cache["resid"][torch.arange(len(b_toks), device=device), b_ends]
        feats = sae.encode(resid_end.to(sae.W_enc.dtype))
        activations.append(feats[:, feat_idx].float().cpu())
    return torch.cat(activations)


def get_feature_with_intervention(toks, ends, ablate_layer, ablate_feat,
                                    measure_layer, measure_feat, batch_size=64):
    #Same idea but with a do(ablate_feat=0) intervention applied at ablate_layer first...
    #...then we measure measure_feat at measure_layer to see what downstream effect it had
    activations = []
    ablate_sae  = saes[ablate_layer]
    measure_sae = saes[measure_layer]
    ablate_hook  = f"blocks.{ablate_layer}.hook_resid_post"
    measure_hook = f"blocks.{measure_layer}.hook_resid_post"

    for i in range(0, len(toks), batch_size):
        b_toks = toks[i:i+batch_size]
        b_ends = ends[i:i+batch_size]
        bsz = len(b_toks)
        batch_idx = torch.arange(bsz, device=device)
        measure_cache = {}

        #Ablation hook: encode -> zero out ablate_feat -> decode -> splice back at end position...
        #...same encode/decode trick as the per-feature AP loop above, just zeroing instead of grad-tracking
        def ablate_hook_fn(act, hook):
            resid_end = act[batch_idx, b_ends]
            feats = ablate_sae.encode(resid_end.to(ablate_sae.W_enc.dtype))
            feats[:, ablate_feat] = 0
            recon_end = ablate_sae.decode(feats).to(act.dtype)
            #Clone before writing so we don't mutate the original act tensor in place
            act = act.clone()
            act[batch_idx, b_ends] = recon_end
            return act

        def measure_hook_fn(act, hook):
            measure_cache["resid"] = act
            return act

        with torch.no_grad():
            _ = model.run_with_hooks(
                b_toks,
                fwd_hooks=[(ablate_hook, ablate_hook_fn),
                            (measure_hook, measure_hook_fn)],
                return_type=None,
            )

        resid_end = measure_cache["resid"][batch_idx, b_ends]
        feats = measure_sae.encode(resid_end.to(measure_sae.W_enc.dtype))
        activations.append(feats[:, measure_feat].float().cpu())
    return torch.cat(activations)

#Now actually run the A -> C edge validation: do(A=0) and see how much C shifts
print("Validating A → C edge:")
print(f"  A = L{L_A}F{f_A}, C = L{L_C}F{f_C}")
c_baseline      = get_feature_activation_at_end(clean_toks, end_positions, L_C, f_C)
c_after_ablateA = get_feature_with_intervention(clean_toks, end_positions, L_A, f_A, L_C, f_C)
#Positive Δ means ablating A pushed C down, so A normally excites C. Negative Δ would mean A inhibits C
delta_C_from_A  = (c_baseline - c_after_ablateA).mean().item()
print(f"  C activation: baseline {c_baseline.mean():.3f} → after do(A=0) {c_after_ablateA.mean():.3f}")
print(f"  Δ = {delta_C_from_A:+.3f}")

Validating A → C edge:
  A = L18F15143, C = L22F10566
  C activation: baseline 19.885 → after do(A=0) 19.006
  Δ = +0.879


In [19]:
# Test direction B → C: does do(B=0) change C?
print("Validating B → C edge:")
print(f"  B = L{L_B}F{f_B}, C = L{L_C}F{f_C}")
c_after_ablateB = get_feature_with_intervention(clean_toks, end_positions, L_B, f_B, L_C, f_C)
delta_C_from_B  = (c_baseline - c_after_ablateB).mean().item()
print(f"  C activation: baseline {c_baseline.mean():.3f} → after do(B=0) {c_after_ablateB.mean():.3f}")
print(f"  Δ = {delta_C_from_B:+.3f}")

# Asymmetry check: does do(C=0) change A or B? Should NOT, if our orientations are right
# (and L_C > L_A, L_B by the layer prior, so C is downstream of both)
print(f"\nReverse-direction sanity check (should be ~0 since C is at a later layer):")
a_baseline      = get_feature_activation_at_end(clean_toks, end_positions, L_A, f_A)
b_baseline      = get_feature_activation_at_end(clean_toks, end_positions, L_B, f_B)

# C is later than A,B in the forward pass — clamping C cannot flow back.
# We run the test to confirm and to demonstrate causal asymmetry.
a_after_ablateC = get_feature_with_intervention(clean_toks, end_positions, L_C, f_C, L_A, f_A)
b_after_ablateC = get_feature_with_intervention(clean_toks, end_positions, L_C, f_C, L_B, f_B)
print(f"  A: baseline {a_baseline.mean():.3f} → after do(C=0) {a_after_ablateC.mean():.3f} "
      f"(Δ {(a_baseline - a_after_ablateC).mean():+.3f})")
print(f"  B: baseline {b_baseline.mean():.3f} → after do(C=0) {b_after_ablateC.mean():.3f} "
      f"(Δ {(b_baseline - b_after_ablateC).mean():+.3f})")

Validating B → C edge:
  B = L18F2297, C = L22F10566
  C activation: baseline 19.885 → after do(B=0) 21.637
  Δ = -1.752

Reverse-direction sanity check (should be ~0 since C is at a later layer):
  A: baseline 8.589 → after do(C=0) 8.589 (Δ +0.000)
  B: baseline 3.758 → after do(C=0) 3.758 (Δ +0.000)


In [ ]:
import random
random.seed(42)

# Filter to cross-layer edges only (within-layer interventions are weird because
# the ablated feature and measured feature are at the same depth)
cross_layer_edges = [(p, c, b) for p, c, b in edges
                     if col_to_layer[p] != col_to_layer[c]]

# Sample up to 50 edges for speed
SAMPLE = min(50, len(cross_layer_edges))
sampled_edges = random.sample(cross_layer_edges, SAMPLE)
print(f"Validating {SAMPLE} cross-layer edges (sampled from {len(cross_layer_edges)})...")

# To save time, we gonna use 500-prompt subset
subsample_toks = clean_toks[:500]
subsample_ends = end_positions[:500]
print(f"  subsample_toks shape: {subsample_toks.shape}, "
      f"subsample_ends shape: {subsample_ends.shape}")

results = []
for idx, (p, c, _) in enumerate(sampled_edges):
    L_p, L_c = col_to_layer[p], col_to_layer[c]
    f_p, f_c = col_to_feat[p], col_to_feat[c]

    baseline = get_feature_activation_at_end(
        subsample_toks, subsample_ends, L_c, f_c, batch_size=64
    )
    intervened = get_feature_with_intervention(
        subsample_toks, subsample_ends, L_p, f_p, L_c, f_c, batch_size=64
    )
    delta = (baseline - intervened).abs().mean().item()

    results.append({
        "parent_col":    p,
        "child_col":     c,
        "L_p":           L_p,
        "L_c":           L_c,
        "f_p":           f_p,
        "f_c":           f_c,
        "baseline_mean": baseline.mean().item(),
        "delta_abs":     delta,
    })
    if (idx + 1) % 10 == 0:
        print(f"  {idx+1}/{SAMPLE} edges validated")


deltas = np.array([r["delta_abs"] for r in results])
THRESHOLD = 0.05   # what counts as "meaningful shift"; tune
pct_validated = (deltas > THRESHOLD).mean() * 100

print(f"\n=== Validation Summary (PC's cross-layer edges, sampled) ===")
print(f"  Sample size:            {SAMPLE}")
print(f"  Mean |Δ|:               {deltas.mean():.4f}")
print(f"  Median |Δ|:             {np.median(deltas):.4f}")
print(f"  Edges with |Δ| > {THRESHOLD}: {pct_validated:.1f}%")

Validating 50 cross-layer edges (sampled from 237)...
  subsample_toks shape: torch.Size([500, 22]), subsample_ends shape: torch.Size([500])
  10/50 edges validated
  20/50 edges validated
  30/50 edges validated
  40/50 edges validated
  50/50 edges validated

=== Validation Summary (PC's cross-layer edges, sampled) ===
  Sample size:            50
  Mean |Δ|:               5.9177
  Median |Δ|:             4.4656
  Edges with |Δ| > 0.05: 100.0%


In [ ]:
torch.save({
    "case_study": {
        "A": {"col": A, "layer": L_A, "feature": f_A, "label": labels[A]["label"],
              "ap_score": ap_for_selected[A].item()},
        "B": {"col": B, "layer": L_B, "feature": f_B, "label": labels[B]["label"],
              "ap_score": ap_for_selected[B].item()},
        "C": {"col": C, "layer": L_C, "feature": f_C, "label": labels[C]["label"],
              "ap_score": ap_for_selected[C].item()},
        "delta_C_from_A": delta_C_from_A,
        "delta_C_from_B": delta_C_from_B,
        "p_marginal_AB":  p_marg,
    },
    "validation_results": results,
    "validation_pct":     pct_validated,
}, DATA_DIR / "case_study_and_validation.pt")
print(f"Saved to {DATA_DIR / 'case_study_and_validation.pt'}")

print("\nThe case study v-structure:")
print(f"  A: {labels[A]['label']}")
print(f"  B: {labels[B]['label']}")
print(f"  C: {labels[C]['label']}")
print(f"\nKey numbers for the report:")
print(f"  Marginal A⊥B p-value: {p_marg:.4f}")
print(f"  do(A=0) → C shift: {delta_C_from_A:+.3f}")
print(f"  do(B=0) → C shift: {delta_C_from_B:+.3f}")
print(f"  PC edge validation rate: {pct_validated:.1f}%")


Saved to /workspace/Independence-Is-All-You-Need/data/case_study_and_validation.pt

=== Day 4 complete ===

The case study v-structure:
  A:  instances of dialogue or reported speech
  B: names of prominent characters or actors associated with performances in films or television shows
  C: specific words and phrases related to arguments, reasoning, and the presentation of evidence in a legal or analytical context

Key numbers for the report:
  Marginal A⊥B p-value: 0.0623
  do(A=0) → C shift: +0.879
  do(B=0) → C shift: -1.752
  PC edge validation rate: 100.0%
